# Task 1: Data Exploration and Enrichment
## Financial Inclusion Forecasting for Ethiopia

**Objective**: Understand the starter dataset and enrich it with additional data useful for forecasting.

**Date**: January 2026

---

## Step 1: Setup and Imports

In [1]:
import sys
import logging
from pathlib import Path
from IPython.display import display

# Add src to path for imports - using absolute path for robustness
src_path = Path.cwd().parent / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Data and analysis
import pandas as pd
import numpy as np
from datetime import datetime

try:
    from data_loader import DataLoader, DataEnricher
    print('✓ Custom modules loaded successfully')
except ImportError as e:
    print(f'✗ Failed to load custom modules: {e}')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print('✓ All imports successful')

✓ Custom modules loaded successfully
✓ All imports successful


## Step 2: Load and Explore Schema

In [3]:
# Initialize DataLoader
loader = DataLoader(data_dir='../data/raw')

# Load datasets
data, reference_codes = loader.load_datasets()

if data is not None:
    print(f'\nDataset Shape: {data.shape}')
    print(f'\nColumn Names: {data.columns.tolist()}')
    display(data.head())
else:
    print('✗ Data failed to load.')

2026-01-31 15:20:10,765 - data_loader - INFO - DataLoader initialized with data_dir: ../data/raw
2026-01-31 15:20:11,039 - data_loader - INFO - Loaded 43 records from ..\data\raw\ethiopia_fi_unified_data.xlsx
2026-01-31 15:20:11,052 - data_loader - INFO - Loaded 14 records from ..\data\raw\ethiopia_fi_unified_data.xlsx
2026-01-31 15:20:11,060 - data_loader - INFO - Loaded reference codes from ..\data\raw\reference_codes.xlsx
2026-01-31 15:20:11,064 - data_loader - INFO - Data validation completed successfully



Dataset Shape: (43, 34)

Column Names: ['record_id', 'record_type', 'category', 'pillar', 'indicator', 'indicator_code', 'indicator_direction', 'value_numeric', 'value_text', 'value_type', 'unit', 'observation_date', 'period_start', 'period_end', 'fiscal_year', 'gender', 'location', 'region', 'source_name', 'source_type', 'source_url', 'confidence', 'related_indicator', 'relationship_type', 'impact_direction', 'impact_magnitude', 'impact_estimate', 'lag_months', 'evidence_basis', 'comparable_country', 'collected_by', 'collection_date', 'original_text', 'notes']


,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,...,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Baseline year,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN


In [4]:
if reference_codes is not None:
    print(f'Reference Codes Shape: {reference_codes.shape}')
    display(reference_codes.head(20))
else:
    print('No reference codes available.')

Reference Codes Shape: (71, 4)


,field,code,description,applies_to
0,record_type,observation,Actual measured value from a source,All
1,record_type,event,Policy launch market event or milestone,All
2,record_type,impact_link,Relationship between event and indicator (link...,All
3,record_type,target,Policy target or official goal,All
4,record_type,baseline,Starting point for comparison,All
5,record_type,forecast,Predicted future value,All
6,category,product_launch,New product or service introduced,event
7,category,market_entry,New competitor enters market,event
8,category,market_exit,Competitor leaves market,event
9,category,policy,Government strategy or regulatory framework,event


In [5]:
# Parse dates for better analysis
data_parsed = loader.parse_dates()

# Ensure datetime type for plotting and range analysis
if 'observation_date' in data_parsed.columns:
    data_parsed['observation_date'] = pd.to_datetime(data_parsed['observation_date'])
    print('✓ Dates parsed successfully')
    print(f'Date range: {data_parsed["observation_date"].min()} to {data_parsed["observation_date"].max()}')

2026-01-31 15:20:24,676 - data_loader - INFO - Date parsing completed


✓ Dates parsed successfully
Date range: 2014-12-31 00:00:00 to 2030-12-31 00:00:00


## Step 3: Understand Data Structure by Record Type

In [6]:
if 'record_type' in data_parsed.columns:
    record_type_counts = data_parsed['record_type'].value_counts()
    print('Records by Type:')
    print(record_type_counts)
    print(f'\nTotal records: {len(data_parsed)}')
else:
    print('record_type column missing.')

Records by Type:
record_type
observation    30
event          10
target          3
Name: count, dtype: int64

Total records: 43


In [7]:
observations = loader.get_observations()
if not observations.empty:
    print(f'\n=== OBSERVATIONS ({len(observations)} records) ===')
    # Check if columns exist before subsetting to prevent KeyError
    cols_to_show = [c for c in ['indicator_code', 'value_numeric', 'observation_date', 'source_type', 'confidence'] if c in observations.columns]
    display(observations[cols_to_show].head(10))
else:
    print('No observations found.')

2026-01-31 15:20:30,507 - data_loader - INFO - Retrieved 30 observation records



=== OBSERVATIONS (30 records) ===


,indicator_code,value_numeric,observation_date,source_type,confidence
0,ACC_OWNERSHIP,22.00,2014-12-31,survey,high
1,ACC_OWNERSHIP,35.00,2017-12-31,survey,high
2,ACC_OWNERSHIP,46.00,2021-12-31,survey,high
3,ACC_OWNERSHIP,56.00,2021-12-31,survey,high
4,ACC_OWNERSHIP,36.00,2021-12-31,survey,high
5,ACC_OWNERSHIP,49.00,2024-11-29,survey,high
6,ACC_MM_ACCOUNT,4.70,2021-12-31,survey,high
7,ACC_MM_ACCOUNT,9.45,2024-11-29,survey,high
8,ACC_4G_COV,37.50,2023-06-30,operator,high
9,ACC_4G_COV,70.80,2025-06-30,operator,high


## Step 4: Analyze Data Coverage and Quality

In [8]:
print('=== TEMPORAL COVERAGE ===')
if not observations.empty and 'observation_date' in observations.columns:
    obs_by_year = observations['observation_date'].dt.year.value_counts().sort_index()
    print(f'\nObservations by Year:\n{obs_by_year}')
else:
    print('No observations with valid dates found.')

=== TEMPORAL COVERAGE ===

Observations by Year:
observation_date
2014     1
2017     1
2021     5
2023     1
2024    11
2025    11
Name: count, dtype: int64


In [9]:
print('\n=== INDICATOR COVERAGE ===')
if not observations.empty:
    indicators = observations.groupby('indicator_code').agg({
        'value_numeric': 'count',
        'observation_date': ['min', 'max']
    })
    indicators.columns = ['Count', 'First', 'Last']
    display(indicators.sort_values('Count', ascending=False))
else:
    print('No indicators found to analyze.')


=== INDICATOR COVERAGE ===


,Count,First,Last
indicator_code,,,
ACC_OWNERSHIP,6,2014-12-31,2024-11-29
ACC_FAYDA,3,2024-08-15,2025-05-15
ACC_4G_COV,2,2023-06-30,2025-06-30
ACC_MM_ACCOUNT,2,2021-12-31,2024-11-29
GEN_GAP_ACC,2,2021-12-31,2024-11-29
USG_P2P_COUNT,2,2024-07-07,2025-07-07
ACC_MOBILE_PEN,1,2025-12-31,2025-12-31
GEN_GAP_MOBILE,1,2024-12-31,2024-12-31
GEN_MM_SHARE,1,2024-12-31,2024-12-31


## Step 5: Data Enrichment

In [10]:
enricher = DataEnricher(data_parsed)
print('Data Enricher initialized.')
print(f'Current record count: {len(enricher.data)}')

2026-01-31 15:20:41,017 - data_loader - INFO - DataEnricher initialized


Data Enricher initialized.
Current record count: 43


In [11]:
enriched_data = enricher.get_enriched_data()
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / 'ethiopia_fi_enriched.csv'
enriched_data.to_csv(output_path, index=False)

print(f'✓ Enriched data saved to {output_path}')

✓ Enriched data saved to ..\data\processed\ethiopia_fi_enriched.csv
